# Claude spot-check, Maersk LLM before-and-after

Semi-manual notebook for the production-interface experiment. Linus pastes
trace fields from the Maersk chatbot UI; the notebook records them, computes
transitions, and emits the side-by-side comparison table.

The query set is fixed and stratified across KB classes. Each ticket is run
twice: once with the old KB, once with the refined KB. `RUN_LABEL` selects
which run is being recorded.

In [ ]:
from __future__ import annotations
import os
import json
from pathlib import Path
from datetime import datetime
import pandas as pd

HERE = Path('.').resolve()
OUTPUT_DIR = HERE / 'dify_exports'
FIGURES_DIR = HERE / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Isolated to claude_implementation/ so the file never collides with Codex outputs.
QUERY_FILE = HERE / 'spot_check_queries.csv'
RUN_LABEL = 'before'  # flip to 'after' for the second pass
SEED = 42

print(f'RUN_LABEL = {RUN_LABEL}')
print(f'Query file: {QUERY_FILE}')


## Step 1, Load or initialise the query set

If `planning/spot_check_queries.csv` does not exist, a 20-row template is
written and the cell exits so Linus can fill it in.

In [ ]:
if not QUERY_FILE.exists():
    template = pd.DataFrame({
        'ticket_id':   [f'Q{i:02d}' for i in range(20)],
        'ticket_text': ['<paste paraphrased ticket text here>'] * 20,
        'class_hint':  [''] * 20,
        'notes':       [''] * 20,
    })
    QUERY_FILE.parent.mkdir(parents=True, exist_ok=True)
    template.to_csv(QUERY_FILE, index=False)
    print(f'Template written to {QUERY_FILE}. Fill it in and re-run this cell.')
    queries = template
else:
    queries = pd.read_csv(QUERY_FILE)
    print(f'Loaded {len(queries)} queries')
    print(queries.head())


## Step 2, Record traces

For each ticket, paste the Maersk chatbot trace into the dict below. Re-run
the cell to append. The cell appends one record at a time so partial runs are
fine.

In [ ]:
def record_trace(ticket_id, retrieved_doc, retrieved_chunk,
                 final_answer, latency_s,
                 correctness, usefulness, comment=''):
    "Append one trace to a per-run CSV. Validates grade enums."
    assert correctness in {'correct', 'partial', 'wrong', 'none'}, correctness
    assert usefulness in {'resolved', 'partial', 'escalation'}, usefulness
    out = OUTPUT_DIR / f'spot_check_{RUN_LABEL}.csv'
    row = {
        'ticket_id': ticket_id,
        'run_label': RUN_LABEL,
        'recorded_at': datetime.now().isoformat(timespec='seconds'),
        'retrieved_doc': retrieved_doc,
        'retrieved_chunk': retrieved_chunk[:1000],
        'final_answer': final_answer[:1000],
        'latency_s': latency_s,
        'correctness': correctness,
        'usefulness': usefulness,
        'comment': comment,
    }
    df_new = pd.DataFrame([row])
    if out.exists():
        df_new = pd.concat([pd.read_csv(out), df_new], ignore_index=True)
    df_new.to_csv(out, index=False)
    print(f'Recorded {ticket_id} ({RUN_LABEL}). Total rows: {len(df_new)}')
    return df_new

# Example, copy this block per ticket and edit
# record_trace(
#     ticket_id='Q00',
#     retrieved_doc='tax/S4_entities.md',
#     retrieved_chunk='S4 entities are reported quarterly...',
#     final_answer='Submit form S4 by EOM.',
#     latency_s=3.2,
#     correctness='correct',
#     usefulness='resolved',
#     comment='Clean retrieval, full answer.',
# )
print('Recording cell ready. Use record_trace() for each ticket.')


## Step 3, Analysis (run after both `before` and `after` are done)

In [ ]:
def load_runs():
    rows = []
    for label in ('before', 'after'):
        f = OUTPUT_DIR / f'spot_check_{label}.csv'
        if f.exists():
            rows.append(pd.read_csv(f))
    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)

def transitions(runs: pd.DataFrame):
    if runs.empty:
        return pd.DataFrame()
    pivot = runs.pivot_table(
        index='ticket_id', columns='run_label',
        values='correctness', aggfunc='first',
    ).fillna('missing')
    pivot['transition'] = pivot.apply(
        lambda r: f'{r.get("before", "missing")} -> {r.get("after", "missing")}', axis=1)
    return pivot

def aggregate_counts(t: pd.DataFrame):
    if 'transition' not in t.columns:
        return {}
    return {
        'no_answer_to_answered':   ((t['before'] == 'none') &
                                    (t['after'].isin(['correct', 'partial']))).sum(),
        'wrong_to_correct':        ((t['before'] == 'wrong') &
                                    (t['after'] == 'correct')).sum(),
        'partial_to_correct':      ((t['before'] == 'partial') &
                                    (t['after'] == 'correct')).sum(),
        'regressions':             ((t['before'] == 'correct') &
                                    (t['after'].isin(['partial', 'wrong', 'none']))).sum(),
        'unchanged_correct':       ((t['before'] == 'correct') &
                                    (t['after'] == 'correct')).sum(),
        'still_no_answer':         ((t['before'] == 'none') &
                                    (t['after'] == 'none')).sum(),
    }

runs = load_runs()
if runs.empty:
    print('No traces recorded yet. Use record_trace() in step 2.')
else:
    t = transitions(runs)
    print('Per-ticket transitions:')
    print(t.to_string())
    print('\nAggregate counts:')
    for k, v in aggregate_counts(t).items():
        print(f'  {k}: {int(v)}')
    t.to_csv(OUTPUT_DIR / 'spot_check_transitions.csv')


## Closing notes

- Two pass run: set `RUN_LABEL = 'before'`, point Maersk chatbot at the old
  KB, record all traces, then set `RUN_LABEL = 'after'`, point at the
  refined KB, record again.
- Aggregate counts feed Act 3 of the paper (`paper_scope_and_narrative.md`).
- Two illustrative traces should be lifted into `assets/figures/` for the
  report. The "S4 entities" chunk-cut-off case is the canonical motivating
  vignette.